In [17]:
import pandas as pd

# Đọc dữ liệu
df = pd.read_csv("air_quality_data_1.csv")

# --- B1. Chuyển cột thời gian về kiểu datetime ---
df["Local Time"] = pd.to_datetime(df["Local Time"], errors="coerce")

# --- B2. Lọc dữ liệu chỉ trong tháng 4/2024 ---
mask = (df["Local Time"].dt.year == 2024) & (df["Local Time"].dt.month == 4)
df_apr2024 = df.loc[mask].copy()

# --- B3. Bảng quy đổi AQI (EPA) ---
PM25_BP = [
    (0.0,   9.0,    0,   50),
    (9.1,  35.4,   51,  100),
    (35.5, 55.4,  101,  150),
    (55.5,125.4,  151,  200),
    (125.5,225.4, 201,  300),
    (225.5,500.4, 301,  500),
]

PM10_BP = [
    (0,   54,   0,   50),
    (55,  154,  51,  100),
    (155, 254, 101,  150),
    (255, 354, 151,  200),
    (355, 424, 201,  300),
    (425, 604, 301,  500),
]

def aqi_from_conc(C, table):
    """Tính AQI theo bảng breakpoints"""
    if pd.isna(C):
        return None
    for BP_lo, BP_hi, I_lo, I_hi in table:
        if BP_lo <= C <= BP_hi:
            return round((I_hi - I_lo) / (BP_hi - BP_lo) * (C - BP_lo) + I_lo)
    return 500 if C > table[-1][1] else 0

# --- B4. Tính lại AQI ---
df_apr2024["AQI_PM25"] = df_apr2024["PM25"].apply(lambda x: aqi_from_conc(x, PM25_BP))
df_apr2024["AQI_PM10"] = df_apr2024["PM10"].apply(lambda x: aqi_from_conc(x, PM10_BP))
df_apr2024["AQI_recalc"] = df_apr2024[["AQI_PM25", "AQI_PM10"]].max(axis=1)

# --- B5. Lưu ra file mới ---
df_apr2024.to_csv("air_quality_AQI_April2024.csv", index=False)

print("✅ Đã tính lại AQI cho tháng 4/2024 và lưu vào file air_quality_AQI_April2024.csv")
print(df_apr2024[["Local Time", "PM25", "PM10", "AQI_PM25", "AQI_PM10", "AQI_recalc"]].head(10))


✅ Đã tính lại AQI cho tháng 4/2024 và lưu vào file air_quality_AQI_April2024.csv
               Local Time   PM25   PM10  AQI_PM25  AQI_PM10  AQI_recalc
10945 2024-04-01 00:00:00  55.50   83.0       151        65         151
10946 2024-04-01 00:00:00  55.50   83.0       151        65         151
10947 2024-04-01 01:00:00  45.83   68.5       126        58         126
10948 2024-04-01 02:00:00  40.14   60.7       112        54         112
10949 2024-04-01 03:00:00  41.33   62.0       115        54         115
10950 2024-04-01 04:00:00  36.67   51.3       104        48         104
10951 2024-04-01 05:00:00  43.00   57.0       119        52         119
10952 2024-04-01 06:00:00  48.00   68.4       132        58         132
10953 2024-04-01 07:00:00  53.06   83.2       144        65         144
10954 2024-04-01 08:00:00  68.00  114.5       160        80         160
